In [1]:
import torch

from notebooks.testing import dataset
from src.model import SequenceCNN

model = SequenceCNN()

model.load_state_dict(
    torch.load(
        "../models/liver_accessibility_cnn.pth"
    )
)

model.eval()

SequenceCNN(
  (conv1): Conv1d(4, 32, kernel_size=(5,), stride=(1,))
  (relu): ReLU()
  (pool): AdaptiveMaxPool1d(output_size=1)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

In [2]:
from src.datasets import SequenceDataset

dataset = SequenceDataset(
    "../data/processed/liver_accessibility.csv"
)

In [3]:
sample_x, sample_y, chrom = dataset[0]

print(sample_x.shape)
print(sample_y)
print(chrom)

torch.Size([4, 200])
tensor([0.])
chr1


In [4]:
sample_x = sample_x.unsqueeze(0)

print(sample_x.shape)

torch.Size([1, 4, 200])


In [5]:
conv_output = model.conv1(sample_x)

print(conv_output.shape)

torch.Size([1, 32, 196])


In [6]:
filter_0_activations = (
    conv_output[0, 0]
    .detach()
    .numpy()
)

print(filter_0_activations.shape)
print(filter_0_activations[:10])

(196,)
[-1.4629786  -1.3532004  -2.8849447  -1.4779568  -0.35114485 -0.5850743
 -0.6914311   0.612103   -1.7311118  -2.9994318 ]


In [17]:
import numpy as np
np.max(filter_0_activations)

0.7682953

In [7]:
max_position = filter_0_activations.argmax()

print(max_position)

115


In [8]:
import pandas as pd

dataset_df = pd.read_csv(
    "../data/processed/liver_accessibility.csv"
)

In [10]:
dataset_df.head()

,chrom,sequence,label
0,chr1,GCAAAGGCGACAGTCCCCACAGTGAAGGGCACTTGTCCATCTGGGG...,0
1,chr1,AAGGAAATATCTTCGTATAAATACTAGACAGAATGATTCTCAGAAA...,0
2,chr3,GTTTGGGGAGTAATTTTAATGCAGAAATACATCATTAATATACTGA...,0
3,chr1,TTACATTTTAGAAAAAACTCTTCTGGCTTACCTAACCTATGTTATG...,0
4,chr3,TTTCTGTTTTTTTATTGCTGTTAATGTTTAACATGAAAATAAGAAT...,1


In [11]:
sequence = dataset_df.iloc[0]["sequence"]

print(sequence)

GCAAAGGCGACAGTCCCCACAGTGAAGGGCACTTGTCCATCTGGGGACTGCTGGTGTTTAGAAGGGGATCTTGGGGCTGCGGCTTTTTCAGAAGGGGACTGGGGTAGAAGGACAACTTCCCCAATATTGTGGGAACACCTGAGTTCCTGCTGGCCGAAGTGAGTACCAAAAATTCTTTGGCAACCAGAAAAGTGACAGCT


In [12]:
motif_candidate = sequence[
    max_position:max_position + 5
]

print(motif_candidate)

CTTCC


#### Checking top-N 5bp motifs per filter

In [13]:
FILTER_INDEX = 0
top_motifs = []

for i in range(1000):

    sample_x, sample_y, chrom = dataset[i]

    sample_x = sample_x.unsqueeze(0)

    conv_output = model.conv1(sample_x)

    activations = (
        conv_output[
            0,
            FILTER_INDEX
        ]
        .detach()
        .numpy()
    )

    max_position = activations.argmax()

    max_activation = activations.max()

    sequence = dataset_df.iloc[i]["sequence"]

    motif_candidate = sequence[
        max_position:max_position + 5
    ]

    top_motifs.append({
        "motif": motif_candidate,
        "activation": max_activation
    })

In [14]:
top_motifs = sorted(
    top_motifs,
    key=lambda x: x["activation"],
    reverse=True
)

In [15]:
for entry in top_motifs[:20]:

    print(
        entry["motif"],
        entry["activation"]
    )

CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
CGTCC 1.3891498
